# Paper 1: Reproduction notebook

**Policy-Invariant Reward Shaping from LLM Feedback: A Framework for Hybrid RL Agents**
Christophe D. Hounwanou, John Emeka Eze, Yaé Ulrich Gaba. 2026.

This notebook is self-contained: every piece of code the paper cites is inlined below, along with runnable demos for each of the paper's data tables.

## What's in this notebook

| Section | Reproduces |
|---|---|
| **A. Setup** | Installs dependencies, detects the LLM backend. |
| **B. Framework code** | All `hybrid/`, `envs/`, and `eval/` modules from the reference implementation. |
| **C. Demo: Proposition 1 numerical verification** | Paper Table 1. Pure Python, no LLM. Runs in about 5 s. |
| **D. Demo: Planner-in-isolation audit** | Paper Table 2. Needs an LLM backend. Runs in 2 to 30 min. |
| **E. Demo: Pipeline validation pilot** | Paper Table 3. Needs an LLM backend and time. Runs in 15 min to 2 h. |

## LLM backends supported

The notebook auto-detects which of these is available and uses the first one it finds. You can also force a specific choice by setting the `LLM_BACKEND` variable manually before running the demos.

1. **Ollama** with `qwen2.5:14b`. The paper's reference configuration. Reproduces Table 2 and Table 3 exactly.
2. **Groq API** with Llama-3.1-8B. Free tier at [console.groq.com](https://console.groq.com), 14,400 requests/day. Reproduces the *framework* but not the exact paper numbers (different LLM, different outputs).
3. **Mock**. Offline hand-scripted responses. For testing notebook mechanics. Does not reproduce real numbers.

## Requirements

- Python 3.10 or newer.
- Internet access for the first pip install.
- Either an Ollama install (with `qwen2.5:14b` pulled) or a `GROQ_API_KEY` environment variable. Skip both if you only want to run demo C.

## How to run

- **Colab / Kaggle:** click `Runtime → Run all`. Set your Groq key in the Secrets panel first if you don't have Ollama.
- **Local Jupyter:** same. If using Ollama, make sure `ollama serve` is running in a separate shell before demos D and E.


## A. Setup


In [ ]:
# Install dependencies. Skip this cell if you already have them.
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "gymnasium==1.2.3",
    "minigrid==3.1.0",
    "stable-baselines3==2.8.0",
    "sentence-transformers==5.2.3",
    "ollama==0.6.2",
    "groq",
    "numpy",
    "tqdm",
    "pandas",
    "pyyaml",
])
print("Dependencies installed.")


### LLM backend selection

The next cell auto-detects which LLM backend is available (Ollama, then Groq, then Mock) and prints its choice. To override, set `LLM_BACKEND` manually before running the demos. See the intro cell for what each backend can reproduce.


In [ ]:
# Auto-detect LLM backend. Sets LLM_BACKEND to one of: "ollama", "groq", "mock".
import os, urllib.request, json as _json

LLM_BACKEND = "mock"  # default fallback
_backend_reason = "no LLM detected"

# Try Ollama with qwen2.5:14b
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as _r:
        _tags = _json.loads(_r.read().decode())
    _models = [m.get("name", "") for m in _tags.get("models", [])]
    if "qwen2.5:14b" in _models:
        LLM_BACKEND = "ollama"
        _backend_reason = "Ollama with qwen2.5:14b (paper's reference config; exact reproduction)"
    else:
        _backend_reason = f"Ollama up but qwen2.5:14b not pulled (models: {_models})"
except Exception:
    pass

# Try Groq if Ollama not available
if LLM_BACKEND == "mock" and os.environ.get("GROQ_API_KEY"):
    LLM_BACKEND = "groq"
    _backend_reason = "Groq API (framework reproduction with Llama-3.1-8B; numbers will differ from paper)"

print(f"LLM_BACKEND = {LLM_BACKEND!r}")
print(f"Reason: {_backend_reason}")
if LLM_BACKEND == "mock":
    print()
    print("To enable a real LLM:")
    print("  Option A (paper-exact): install Ollama and run: ollama pull qwen2.5:14b")
    print("  Option B (framework):   get a free Groq key at https://console.groq.com,")
    print("                          then set GROQ_API_KEY environment variable.")


## B. Framework code

The following cells inline every module the paper describes. Run them in order — later cells depend on classes and functions defined in earlier ones.


### B1. Prompt templates (paper Appendix A: P1–P4)

The four prompt templates the LLM planner uses: P1 planning, P2 scoring (for the potential function $\Phi$), P3 completion oracle, P4 failure-triggered replanning. Verbatim; not paraphrased anywhere in the pipeline.


In [ ]:
"""Prompt templates P1-P4 from paper Appendix A. Verbatim strings; do not paraphrase.

If a prompt changes here, the paper's Appendix A must change to match.
"""
from __future__ import annotations

from dataclasses import dataclass

P1_PLANNING_SYSTEM = (
    "You are a planner for an agent operating in {environment_name}. "
    "The agent receives a task in natural language and must complete it "
    "through a sequence of subgoals. Emit a list of at most {max_plan_length} "
    "subgoals, one per line, each a short imperative English clause of at "
    "most {max_subgoal_tokens} tokens."
)
P1_PLANNING_USER = (
    "Task: {task_string}\n"
    "Current state: {state_caption}\n"
    "\n"
    "Output:"
)

P2_SCORING_SYSTEM = (
    "You are a progress estimator. Given a subgoal and the current state, "
    "output a single number in [0, 1] estimating how close the agent is to "
    "completing the subgoal. Output only the number."
)
P2_SCORING_USER = (
    "Subgoal: {subgoal}\n"
    "State: {state_caption}\n"
    "\n"
    "Score:"
)

P3_COMPLETION_SYSTEM = (
    "Given a subgoal and the current state, output exactly one token: "
    "DONE if the subgoal is now satisfied, CONTINUE otherwise."
)
P3_COMPLETION_USER = (
    "Subgoal: {subgoal}\n"
    "State: {state_caption}\n"
    "\n"
    "Answer:"
)

P4_REPLAN_SYSTEM = (
    "You are revising a plan mid-execution. The previous subgoal failed to "
    "complete within its step budget. Revise the remaining plan. Emit a list "
    "of at most {max_plan_length} subgoals."
)
P4_REPLAN_USER = (
    "Original task: {task_string}\n"
    "Current state: {state_caption}\n"
    "Failed subgoal: {subgoal}\n"
    "Reason: exceeded step budget of {B} steps.\n"
    "\n"
    "Revised plan:"
)

@dataclass
class PromptContext:
    """Field container used by planner backends to fill any of P1-P4."""
    environment_name: str = "MiniGrid"
    max_plan_length: int = 12
    max_subgoal_tokens: int = 32
    task_string: str = ""
    state_caption: str = ""
    subgoal: str = ""
    B: int = 25

    def fill(self, template: str) -> str:
        return template.format(**self.__dict__)


### B2. Planner backends (Ollama, Groq, Mock, plus parsers)

Three implementations of the `Planner` protocol. Ollama and Groq are real LLM backends; Mock is a deterministic offline stand-in for tests. Also includes `parse_plan` and `parse_score` — the parsers that turn LLM outputs into subgoal lists and $[0, 1]$ scores.


In [ ]:
"""LLM planner client. Two backends: Ollama (real) and Mock (deterministic, offline).

The Planner interface has three methods:
    plan(task, state_caption) -> list[str]        (P1 or P4)
    score(subgoal, state_caption) -> float in [0,1] (P2)
    completed(subgoal, state_caption) -> bool     (P3)

Both backends implement all three. MockPlanner uses hand-scripted responses
for the standard MiniGrid task families so tests can run without Ollama.
"""
from __future__ import annotations

import re
import time
from dataclasses import dataclass, field
from typing import Protocol

# (defined in an earlier cell)

@dataclass
class PlannerStats:
    """Cumulative counters — logged per-episode in train.py."""
    calls: int = 0
    tokens_in: int = 0
    tokens_out: int = 0
    wallclock_sec: float = 0.0

class Planner(Protocol):
    stats: PlannerStats

    def plan(self, task: str, state_caption: str,
             failed_subgoal: str | None = None) -> list[str]: ...
    def score(self, subgoal: str, state_caption: str) -> float: ...
    def completed(self, subgoal: str, state_caption: str) -> bool: ...

# --------------------------------------------------------------------------
# Ollama backend
# --------------------------------------------------------------------------

class OllamaPlanner:
    """Real LLM planner served by a local Ollama daemon."""

    def __init__(self, model: str = "qwen2.5:14b",
                 max_plan_length: int = 12,
                 max_subgoal_tokens: int = 32,
                 environment_name: str = "MiniGrid",
                 temperature: float = 0.0,
                 host: str | None = None):
        try:
            import ollama
        except ImportError as e:
            raise RuntimeError(
                "ollama Python package not installed. `pip install ollama`."
            ) from e
        self.model = model
        self.max_plan_length = max_plan_length
        self.max_subgoal_tokens = max_subgoal_tokens
        self.environment_name = environment_name
        self.temperature = temperature
        self._client = ollama.Client(host=host) if host else ollama.Client()
        self.stats = PlannerStats()

    def _chat(self, system: str, user: str, num_predict: int = 256) -> tuple[str, int, int]:
        t0 = time.perf_counter()
        resp = self._client.chat(
            model=self.model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            options={
                "temperature": self.temperature,
                "num_predict": num_predict,
            },
        )
        elapsed = time.perf_counter() - t0
        content = resp["message"]["content"]
        # Ollama returns eval_count / prompt_eval_count in the response
        toks_in = int(resp.get("prompt_eval_count", 0))
        toks_out = int(resp.get("eval_count", 0))
        self.stats.calls += 1
        self.stats.tokens_in += toks_in
        self.stats.tokens_out += toks_out
        self.stats.wallclock_sec += elapsed
        return content, toks_in, toks_out

    def _ctx(self, **kw) -> PromptContext:
        return PromptContext(
            environment_name=self.environment_name,
            max_plan_length=self.max_plan_length,
            max_subgoal_tokens=self.max_subgoal_tokens,
            **kw,
        )

    def plan(self, task: str, state_caption: str,
             failed_subgoal: str | None = None) -> list[str]:
        if failed_subgoal is None:
            ctx = self._ctx(task_string=task, state_caption=state_caption)
            content, _, _ = self._chat(
                ctx.fill(P1_PLANNING_SYSTEM),
                ctx.fill(P1_PLANNING_USER),
                num_predict=256,
            )
        else:
            ctx = self._ctx(task_string=task, state_caption=state_caption,
                            subgoal=failed_subgoal)
            content, _, _ = self._chat(
                ctx.fill(P4_REPLAN_SYSTEM),
                ctx.fill(P4_REPLAN_USER),
                num_predict=256,
            )
        return parse_plan(content, max_plan_length=self.max_plan_length,
                          max_subgoal_tokens=self.max_subgoal_tokens)

    def score(self, subgoal: str, state_caption: str) -> float:
        ctx = self._ctx(subgoal=subgoal, state_caption=state_caption)
        content, _, _ = self._chat(
            ctx.fill(P2_SCORING_SYSTEM),
            ctx.fill(P2_SCORING_USER),
            num_predict=8,
        )
        return parse_score(content)

    def completed(self, subgoal: str, state_caption: str) -> bool:
        ctx = self._ctx(subgoal=subgoal, state_caption=state_caption)
        content, _, _ = self._chat(
            ctx.fill(P3_COMPLETION_SYSTEM),
            ctx.fill(P3_COMPLETION_USER),
            num_predict=4,
        )
        return "DONE" in content.upper()

# --------------------------------------------------------------------------
# Groq backend (OpenAI-compatible API, free tier)
# --------------------------------------------------------------------------

class GroqPlanner:
    """Real LLM planner served by Groq's OpenAI-compatible API.

    Reads GROQ_API_KEY from the environment. Free tier as of 2026: ~14,400
    requests/day on Llama-3.1-8B, generous throughput. Useful when Ollama is
    unavailable (e.g. Colab, Kaggle) and the exact-LLM reproduction of the
    paper's tables is not required.

    Note: the paper's Table 2 numbers are specific to Qwen-2.5:14b via Ollama.
    Reproducing the framework with Groq's Llama-3.1 will produce comparable but
    different numbers --- that is a legitimate reproduction of the framework,
    not of the paper's exact planner-audit measurements.
    """

    def __init__(self, model: str = "llama-3.1-8b-instant",
                 max_plan_length: int = 12,
                 max_subgoal_tokens: int = 32,
                 environment_name: str = "MiniGrid",
                 temperature: float = 0.0,
                 api_key: str | None = None):
        try:
            from groq import Groq
        except ImportError as e:
            raise RuntimeError(
                "groq Python package not installed. `pip install groq`."
            ) from e
        import os
        key = api_key or os.environ.get("GROQ_API_KEY")
        if not key:
            raise RuntimeError(
                "GROQ_API_KEY not set. Get a free key at https://console.groq.com "
                "and set it as an environment variable."
            )
        self.model = model
        self.max_plan_length = max_plan_length
        self.max_subgoal_tokens = max_subgoal_tokens
        self.environment_name = environment_name
        self.temperature = temperature
        self._client = Groq(api_key=key)
        self.stats = PlannerStats()

    def _chat(self, system: str, user: str, num_predict: int = 256) -> tuple[str, int, int]:
        t0 = time.perf_counter()
        resp = self._client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            temperature=self.temperature,
            max_tokens=num_predict,
        )
        elapsed = time.perf_counter() - t0
        content = resp.choices[0].message.content or ""
        # Groq's response.usage matches OpenAI's schema
        toks_in = int(getattr(resp.usage, "prompt_tokens", 0)) if resp.usage else 0
        toks_out = int(getattr(resp.usage, "completion_tokens", 0)) if resp.usage else 0
        self.stats.calls += 1
        self.stats.tokens_in += toks_in
        self.stats.tokens_out += toks_out
        self.stats.wallclock_sec += elapsed
        return content, toks_in, toks_out

    def _ctx(self, **kw) -> PromptContext:
        return PromptContext(
            environment_name=self.environment_name,
            max_plan_length=self.max_plan_length,
            max_subgoal_tokens=self.max_subgoal_tokens,
            **kw,
        )

    def plan(self, task: str, state_caption: str,
             failed_subgoal: str | None = None) -> list[str]:
        if failed_subgoal is None:
            ctx = self._ctx(task_string=task, state_caption=state_caption)
            content, _, _ = self._chat(
                ctx.fill(P1_PLANNING_SYSTEM),
                ctx.fill(P1_PLANNING_USER),
                num_predict=256,
            )
        else:
            ctx = self._ctx(task_string=task, state_caption=state_caption,
                            subgoal=failed_subgoal)
            content, _, _ = self._chat(
                ctx.fill(P4_REPLAN_SYSTEM),
                ctx.fill(P4_REPLAN_USER),
                num_predict=256,
            )
        return parse_plan(content, max_plan_length=self.max_plan_length,
                          max_subgoal_tokens=self.max_subgoal_tokens)

    def score(self, subgoal: str, state_caption: str) -> float:
        ctx = self._ctx(subgoal=subgoal, state_caption=state_caption)
        content, _, _ = self._chat(
            ctx.fill(P2_SCORING_SYSTEM),
            ctx.fill(P2_SCORING_USER),
            num_predict=8,
        )
        return parse_score(content)

    def completed(self, subgoal: str, state_caption: str) -> bool:
        ctx = self._ctx(subgoal=subgoal, state_caption=state_caption)
        content, _, _ = self._chat(
            ctx.fill(P3_COMPLETION_SYSTEM),
            ctx.fill(P3_COMPLETION_USER),
            num_predict=4,
        )
        return "DONE" in content.upper()

# --------------------------------------------------------------------------
# Mock backend — offline, deterministic, for tests
# --------------------------------------------------------------------------

# Hand-scripted plans keyed by MiniGrid task-family regex.
# Each entry: (regex, plan) — the first match wins.
_MOCK_PLANS: list[tuple[re.Pattern, list[str]]] = [
    (re.compile(r"unlock.*door|use.*key.*door|key.*door", re.I),
     ["go to the key", "pick up the key", "go to the door",
      "unlock the door", "go to the goal"]),
    (re.compile(r"pick up.*(?:key|ball|box)", re.I),
     ["go to the target object", "pick up the target object"]),
    (re.compile(r"go to.*goal|reach.*goal", re.I),
     ["go to the goal"]),
    (re.compile(r"go to.*(?:key|ball|box|door)", re.I),
     ["go to the target object"]),
]

_DEFAULT_MOCK_PLAN = ["explore the environment", "go to the goal"]

@dataclass
class MockPlanner:
    """Deterministic offline planner. Returns hand-scripted plans by task regex.
    score() and completed() use string heuristics on the state caption."""
    max_plan_length: int = 12
    max_subgoal_tokens: int = 32
    environment_name: str = "MiniGrid"
    stats: PlannerStats = field(default_factory=PlannerStats)

    def plan(self, task: str, state_caption: str,
             failed_subgoal: str | None = None) -> list[str]:
        self.stats.calls += 1
        for rx, p in _MOCK_PLANS:
            if rx.search(task):
                return p[:self.max_plan_length]
        return list(_DEFAULT_MOCK_PLAN)

    def score(self, subgoal: str, state_caption: str) -> float:
        """Cheap heuristic: fraction of content-word hits in the state caption."""
        self.stats.calls += 1
        stopwords = {"the", "a", "an", "to", "of", "at", "in", "on", "up", "and", "or"}
        words = [w.lower() for w in re.findall(r"\w+", subgoal)]
        keywords = [w for w in words if len(w) >= 3 and w not in stopwords]
        if not keywords:
            return 0.5
        caption_lower = state_caption.lower()
        hits = sum(1 for k in keywords if k in caption_lower)
        return min(1.0, hits / len(keywords))

    def completed(self, subgoal: str, state_caption: str) -> bool:
        """Return True if a strong lexical match for subgoal appears in caption."""
        self.stats.calls += 1
        s = subgoal.lower()
        c = state_caption.lower()
        # e.g. "pick up the key" satisfied when caption says "carrying key"
        if "pick up" in s or "carry" in s:
            obj = re.sub(r"pick up|the|a|an|carry", "", s).strip()
            return f"carrying {obj}" in c
        if "unlock" in s:
            return "door open" in c or "unlocked" in c
        if "go to" in s or "reach" in s:
            obj = re.sub(r"go to|reach|the|a|an", "", s).strip()
            return f"at {obj}" in c or f"on {obj}" in c
        return False

# --------------------------------------------------------------------------
# Parsers
# --------------------------------------------------------------------------

def parse_plan(text: str, max_plan_length: int = 12,
               max_subgoal_tokens: int = 32) -> list[str]:
    """Parse an LLM plan output into a list of subgoal strings.

    Handles:
      - numbered lists (1. foo, 1) foo)
      - bullet lists (- foo, * foo)
      - plain newline-separated lines
    Empty lines and lines with no alpha characters are dropped.
    """
    out: list[str] = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        # strip common list prefixes
        line = re.sub(r"^\s*(?:\d+[\.\)]\s*|[\-\*\+]\s*)", "", line)
        line = line.strip()
        if not line or not re.search(r"[A-Za-z]", line):
            continue
        # truncate to token limit (rough word-based approximation)
        toks = line.split()
        if len(toks) > max_subgoal_tokens:
            line = " ".join(toks[:max_subgoal_tokens])
        out.append(line)
        if len(out) >= max_plan_length:
            break
    return out

def parse_score(text: str) -> float:
    """Parse a scoring-prompt response into a [0, 1] float. Returns 0.5 on failure."""
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", text)
    if not m:
        return 0.5
    try:
        v = float(m.group(0))
    except ValueError:
        return 0.5
    return max(0.0, min(1.0, v))


### B3. Potential-based reward shaping (paper §4.5, Proposition 1)

Implements $F(s, g, s') = \gamma \Phi(s', g) - \Phi(s, g)$ where $\Phi$ is the LLM's clamped $[0, 1]$ score. This is the piece Proposition 1 applies to: because $F$ is a telescoping difference of a bounded potential, the shaping term does not alter the set of optimal policies of the augmented MDP.


In [ ]:
"""Potential-based reward shaping from LLM feedback.

Given a bounded potential Phi : S x G -> [0, 1], the shaping term
    F(s, g, s') = gamma * Phi(s', g) - Phi(s, g)
preserves the set of optimal policies of the augmented MDP
(Ng, Harada, Russell 1999; paper Proposition 1).

Implementation caches Phi(s, g) evaluations so we don't call the LLM twice
for the same (state_caption, subgoal) pair.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Hashable

# (defined in an earlier cell)

@dataclass
class PotentialShaper:
    """Wraps a Planner as a bounded potential function.

    magnitude c > 0 scales the LLM score into Phi. Default c=1.0 keeps Phi in [0,1].
    """
    planner: Planner
    gamma: float = 0.99
    magnitude: float = 1.0
    enabled: bool = True
    _cache: dict[Hashable, float] = field(default_factory=dict)

    def phi(self, state_caption: str, subgoal: str) -> float:
        if not self.enabled:
            return 0.0
        key = (state_caption, subgoal)
        cached = self._cache.get(key)
        if cached is not None:
            return cached
        s = self.planner.score(subgoal, state_caption)
        # Clamp to [0, 1] defensively; parse_score already clamps but Phi
        # must be bounded for Prop 1 to apply.
        s = max(0.0, min(1.0, float(s)))
        v = self.magnitude * s
        self._cache[key] = v
        return v

    def shape(self, state_caption: str, next_caption: str, subgoal: str) -> float:
        """Return F(s, g, s') = gamma * Phi(s', g) - Phi(s, g)."""
        if not self.enabled:
            return 0.0
        phi_curr = self.phi(state_caption, subgoal)
        phi_next = self.phi(next_caption, subgoal)
        return self.gamma * phi_next - phi_curr

# --------------------------------------------------------------------------
# Analytic potential — used by tests, verify/prop1_numerical.py, and any
# ablation that swaps out the LLM for a hand-designed Phi.
# --------------------------------------------------------------------------

@dataclass
class AnalyticShaper:
    """Wraps a caller-provided Phi function. Used to test Prop 1 with arbitrary Phi."""
    phi_fn: Callable[[Hashable, Hashable], float]
    gamma: float = 0.99
    magnitude: float = 1.0
    enabled: bool = True

    def phi(self, state, subgoal) -> float:
        if not self.enabled:
            return 0.0
        return self.magnitude * float(self.phi_fn(state, subgoal))

    def shape(self, state, next_state, subgoal) -> float:
        if not self.enabled:
            return 0.0
        return self.gamma * self.phi(next_state, subgoal) - self.phi(state, subgoal)


### B4. Subgoal scheduler + `Done` oracles (paper §4.3)

Maintains the active-subgoal pointer $k_t$ and the per-subgoal budget counter $b_t$. Ships three implementations of the completion oracle $\mathrm{Done}$: environment event flags, a learned MLP classifier, and an LLM completion prompt.


In [ ]:
"""Subgoal scheduling: three Done-oracle variants + pointer advancement.

Done : S x G -> {0, 1} — advances the active-subgoal pointer when it fires.

Three implementations per paper §4.3:
  (i)   EnvEventDone   — environment provides an event flag (fast, reliable when available)
  (ii)  MLPDone        — small learned classifier over state + subgoal embedding
                          (stub here; training loop not implemented in the CPU repo)
  (iii) LLMDone        — LLM is prompted with P3 and outputs DONE/CONTINUE

Each variant conforms to the DoneOracle Protocol.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Protocol

# (defined in an earlier cell)

class DoneOracle(Protocol):
    def __call__(self, state, next_state, next_caption: str, subgoal: str) -> bool: ...

# --------------------------------------------------------------------------
# EnvEventDone: caller supplies a function state -> {event_name: bool}
# --------------------------------------------------------------------------

@dataclass
class EnvEventDone:
    """Uses environment-provided event flags. The caller registers a mapping
    from subgoal string -> event key. If subgoal not registered, returns False."""
    event_extractor: Callable[[object], dict[str, bool]]
    subgoal_to_event: dict[str, str] = field(default_factory=dict)

    def register(self, subgoal: str, event: str) -> None:
        self.subgoal_to_event[subgoal] = event

    def __call__(self, state, next_state, next_caption: str, subgoal: str) -> bool:
        event = self.subgoal_to_event.get(subgoal)
        if event is None:
            # Fall back to substring match against caption (best effort)
            return _caption_matches_subgoal(next_caption, subgoal)
        flags = self.event_extractor(next_state)
        return bool(flags.get(event, False))

def _caption_matches_subgoal(caption: str, subgoal: str) -> bool:
    """Best-effort substring check. Matches paper's MockPlanner heuristics."""
    import re
    s = subgoal.lower()
    c = caption.lower()
    if "pick up" in s or "carry" in s:
        obj = re.sub(r"pick up|the|a|an|carry", "", s).strip()
        return f"carrying {obj}" in c
    if "unlock" in s:
        return "door open" in c or "unlocked" in c
    if "go to" in s or "reach" in s:
        obj = re.sub(r"go to|reach|the|a|an", "", s).strip()
        return f"at {obj}" in c or f"on {obj}" in c
    return False

# --------------------------------------------------------------------------
# LLMDone: uses P3 prompt via Planner.completed()
# --------------------------------------------------------------------------

@dataclass
class LLMDone:
    planner: Planner

    def __call__(self, state, next_state, next_caption: str, subgoal: str) -> bool:
        return self.planner.completed(subgoal, next_caption)

# --------------------------------------------------------------------------
# MLPDone: stub — the classifier training loop is out of scope for the CPU
# reproducibility repo. Documented for the GPU rebuild; falls back to caption
# match at inference time here.
# --------------------------------------------------------------------------

@dataclass
class MLPDone:
    """Learned MLP over (state_embed, subgoal_embed) -> {0,1}. Stub only.

    In the full pipeline this would be a small torch model trained on a
    labeled dataset of (state, subgoal, done) tuples. The CPU repo does not
    include the training loop; the ablation config that selects this oracle
    will log a warning and fall back to caption-matching at inference.
    """
    def __call__(self, state, next_state, next_caption: str, subgoal: str) -> bool:
        import warnings
        warnings.warn(
            "MLPDone stub: no trained classifier available in the CPU repo. "
            "Falling back to caption-substring matching. To use MLPDone for "
            "real, implement the classifier training loop and load its weights.",
            RuntimeWarning, stacklevel=2,
        )
        return _caption_matches_subgoal(next_caption, subgoal)

# --------------------------------------------------------------------------
# SubgoalScheduler: pointer advancement + budget counter
# --------------------------------------------------------------------------

@dataclass
class SubgoalScheduler:
    """Holds the plan, the active-subgoal pointer k, and the per-subgoal budget counter b."""
    plan: list[str]
    oracle: DoneOracle
    k: int = 0
    b: int = 0

    @property
    def active(self) -> str:
        if not self.plan:
            return ""
        return self.plan[min(self.k, len(self.plan) - 1)]

    def step(self, state, next_state, next_caption: str) -> dict:
        """Advance or increment. Returns diagnostics dict.

        Returns:
            {'advanced': bool, 'k': int, 'b': int, 'active_subgoal': str}
        """
        done = self.oracle(state, next_state, next_caption, self.active)
        if done:
            self.k = min(self.k + 1, max(0, len(self.plan) - 1))
            self.b = 0
            advanced = True
        else:
            self.b += 1
            advanced = False
        return dict(advanced=advanced, k=self.k, b=self.b,
                    active_subgoal=self.active)

    def replace_from_current(self, new_suffix: list[str]) -> None:
        """Replace plan from current pointer position onward (used on replan)."""
        self.plan = self.plan[:self.k] + new_suffix
        self.b = 0


### B5. Replan trigger (paper §4.6)

Fires the LLM planner mid-episode under two conditions: periodic (every $H$ steps) and failure (per-subgoal budget $b_t \geq B$ without $\mathrm{Done}$ firing).


In [ ]:
"""Replanning triggers: periodic (every H steps) and failure (budget b >= B).

Both are stateless with respect to policy learning; they only affect the
plan currently held by the SubgoalScheduler.
"""
from __future__ import annotations

from dataclasses import dataclass

# (defined in an earlier cell)
# (defined in an earlier cell)

@dataclass
class ReplanTrigger:
    planner: Planner
    period_H: int = 50           # periodic replan every H steps (0 disables)
    budget_B: int = 25           # per-subgoal budget for failure trigger (0 disables)

    def maybe_replan(self,
                     t: int,
                     task: str,
                     state_caption: str,
                     scheduler: SubgoalScheduler) -> tuple[bool, str | None]:
        """Called every environment step. Returns (fired, reason).

        Reason is one of {"periodic", "failure", None}.
        """
        # Failure trigger fires first — if the current subgoal is stuck,
        # we don't want to wait for the periodic tick.
        if self.budget_B > 0 and scheduler.b >= self.budget_B:
            new_suffix = self.planner.plan(
                task, state_caption, failed_subgoal=scheduler.active,
            )
            scheduler.replace_from_current(new_suffix)
            return True, "failure"

        if self.period_H > 0 and t > 0 and (t % self.period_H == 0):
            new_suffix = self.planner.plan(task, state_caption)
            scheduler.replace_from_current(new_suffix)
            return True, "periodic"

        return False, None


### B6. Subgoal encoder + gym wrapper (paper §4.4)

Encodes each subgoal string into a fixed-length vector (frozen sentence transformer by default) and augments the environment's observation dict with a `subgoal_embedding` key that the RL policy conditions on.


In [ ]:
"""Subgoal-conditioned policy pieces: subgoal encoder + gym wrapper.

The policy itself is a stable-baselines3 PPO with MultiInputPolicy over the
Dict observation produced by SubgoalConditionedWrapper. The wrapper adds a
'subgoal_embedding' key to the observation dict; the base env's observation
is preserved under 'obs' (flattened via MiniGrid's FlatObsWrapper if
applicable, done at env construction time).
"""
from __future__ import annotations

import hashlib
from dataclasses import dataclass
from typing import Protocol

import numpy as np

class SubgoalEncoder(Protocol):
    embedding_dim: int
    def encode(self, subgoal: str) -> np.ndarray: ...

# --------------------------------------------------------------------------
# Real encoder — sentence-transformers MiniLM-L6-v2 (paper §4.4)
# --------------------------------------------------------------------------

class STEncoder:
    """Frozen sentence-transformer. Default: all-MiniLM-L6-v2 (dim=384)."""

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError as e:
            raise RuntimeError(
                "sentence-transformers not installed. `pip install sentence-transformers`."
            ) from e
        self._model = SentenceTransformer(model_name)
        self.embedding_dim = int(self._model.get_sentence_embedding_dimension())
        self._cache: dict[str, np.ndarray] = {}

    def encode(self, subgoal: str) -> np.ndarray:
        cached = self._cache.get(subgoal)
        if cached is not None:
            return cached
        v = self._model.encode([subgoal], convert_to_numpy=True,
                               normalize_embeddings=True)[0].astype(np.float32)
        self._cache[subgoal] = v
        return v

# --------------------------------------------------------------------------
# Mock encoder — deterministic, offline, no torch model load
# --------------------------------------------------------------------------

@dataclass
class MockEncoder:
    """Deterministic hash-based embedding for tests. Same string -> same vector."""
    embedding_dim: int = 64

    def encode(self, subgoal: str) -> np.ndarray:
        # SHA256 of the subgoal -> repeat/truncate to embedding_dim bytes
        h = hashlib.sha256(subgoal.encode("utf-8")).digest()
        needed = self.embedding_dim
        raw = (h * ((needed // len(h)) + 1))[:needed]
        # bytes -> [0, 1] -> [-1, 1]
        arr = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) / 255.0) * 2 - 1
        # normalize
        n = np.linalg.norm(arr)
        if n > 0:
            arr = arr / n
        return arr

# --------------------------------------------------------------------------
# Gym wrapper: augments observation dict with 'subgoal_embedding'
# --------------------------------------------------------------------------

def make_subgoal_wrapper(env, encoder: SubgoalEncoder):
    """Returns a gymnasium.Wrapper that exposes a Dict observation with keys
    'obs' (base env's observation) and 'subgoal_embedding'."""
    import gymnasium as gym
    from gymnasium import spaces

    class _SubgoalConditionedWrapper(gym.Wrapper):
        def __init__(self, env, encoder):
            super().__init__(env)
            self._encoder = encoder
            self._subgoal: str = ""
            self._embedding: np.ndarray = np.zeros(
                encoder.embedding_dim, dtype=np.float32,
            )
            self.observation_space = spaces.Dict({
                "obs": env.observation_space,
                "subgoal_embedding": spaces.Box(
                    low=-1.0, high=1.0,
                    shape=(encoder.embedding_dim,), dtype=np.float32,
                ),
            })

        # Called by the training loop when the active subgoal changes.
        def set_subgoal(self, subgoal: str) -> None:
            if subgoal == self._subgoal:
                return
            self._subgoal = subgoal
            self._embedding = self._encoder.encode(subgoal).astype(np.float32)

        @property
        def current_subgoal(self) -> str:
            return self._subgoal

        def reset(self, **kwargs):
            obs, info = self.env.reset(**kwargs)
            return self._obs_dict(obs), info

        def step(self, action):
            obs, reward, terminated, truncated, info = self.env.step(action)
            return self._obs_dict(obs), reward, terminated, truncated, info

        def _obs_dict(self, obs):
            return {"obs": obs, "subgoal_embedding": self._embedding.copy()}

    return _SubgoalConditionedWrapper(env, encoder)


### B7. MiniGrid environment helpers

Two utilities the pipeline uses on MiniGrid environments: `caption_minigrid(env)` produces the text description $\phi(s)$ from the underlying grid, and `extract_events(env)` returns boolean flags for `carrying_key`, `any_door_open`, `at_goal`, etc., that the env-event $\mathrm{Done}$ oracle keys on.


In [ ]:
"""MiniGrid observation → text captioner φ + environment-event extractor.

MiniGrid observations expose:
  - obs['image']    : (H, W, 3) view of the agent's field of view (encoded)
  - obs['direction']: int 0-3 (agent facing direction)
  - obs['mission']  : natural-language task string

For φ we produce a compact text caption from the full grid (via env.unwrapped.grid),
agent position, agent direction, and what the agent is carrying. This is a lossy but
faithful summary of the state — sufficient for the planner and the LLM Done oracle.

For the env-event extractor, MiniGrid tracks: agent position, carrying, door.is_open,
which we surface as boolean flags keyed by natural-language event names.
"""
from __future__ import annotations

from typing import Callable

import numpy as np

_DIRECTIONS = ["right", "down", "left", "up"]

def caption_minigrid(env) -> str:
    """Text summary of the current MiniGrid state, computed from the underlying grid.

    Format example (DoorKey-6x6):
        "6x6 room. Agent at (1, 4) facing up. Carrying: nothing.
         Objects: yellow key at (3, 1), yellow door at (4, 2) closed and locked, green goal at (5, 4)."
    """
    unwrapped = env.unwrapped
    grid = unwrapped.grid
    w, h = grid.width, grid.height
    pos = getattr(unwrapped, "agent_pos", None)
    d = getattr(unwrapped, "agent_dir", None)
    carrying = getattr(unwrapped, "carrying", None)

    parts = [f"{w}x{h} room."]
    if pos is not None and d is not None:
        parts.append(f"Agent at {tuple(int(x) for x in pos)} facing {_DIRECTIONS[int(d)]}.")
    if carrying is None or getattr(carrying, "type", None) is None:
        parts.append("Carrying: nothing.")
    else:
        color = getattr(carrying, "color", "?")
        parts.append(f"Carrying: {color} {carrying.type}.")

    objs: list[str] = []
    for i in range(w):
        for j in range(h):
            cell = grid.get(i, j)
            if cell is None or cell.type in ("wall", "floor", "unseen"):
                continue
            desc = f"{cell.color} {cell.type} at ({i}, {j})"
            if cell.type == "door":
                is_open = getattr(cell, "is_open", False)
                is_locked = getattr(cell, "is_locked", False)
                state = "open" if is_open else ("closed and locked" if is_locked else "closed")
                desc += f" {state}"
            objs.append(desc)
    if objs:
        parts.append("Objects: " + ", ".join(objs) + ".")
    return " ".join(parts)

def extract_events(env) -> dict[str, bool]:
    """Environment-provided event flags. Keys are natural-language event names."""
    unwrapped = env.unwrapped
    carrying = getattr(unwrapped, "carrying", None)
    events: dict[str, bool] = {
        "carrying_key": (carrying is not None and getattr(carrying, "type", "") == "key"),
        "carrying_ball": (carrying is not None and getattr(carrying, "type", "") == "ball"),
        "carrying_box": (carrying is not None and getattr(carrying, "type", "") == "box"),
    }
    # any door open in the grid
    grid = unwrapped.grid
    any_door_open = False
    at_goal = False
    pos = getattr(unwrapped, "agent_pos", None)
    for i in range(grid.width):
        for j in range(grid.height):
            cell = grid.get(i, j)
            if cell is None:
                continue
            if cell.type == "door" and getattr(cell, "is_open", False):
                any_door_open = True
            if cell.type == "goal" and pos is not None and (int(pos[0]) == i and int(pos[1]) == j):
                at_goal = True
    events["any_door_open"] = any_door_open
    events["at_goal"] = at_goal
    return events

def make_minigrid_captioner() -> Callable[[object], str]:
    return caption_minigrid

def make_minigrid_event_extractor() -> Callable[[object], dict[str, bool]]:
    return extract_events

# --------------------------------------------------------------------------
# Default subgoal → event mapping for the common MiniGrid task families
# --------------------------------------------------------------------------

DEFAULT_SUBGOAL_TO_EVENT: dict[str, str] = {
    # Common subgoal phrasings from mock/LLM plans → event flag from extract_events
    "go to the key": "at_key_placeholder",           # captured via caption fallback
    "pick up the key": "carrying_key",
    "go to the door": "at_door_placeholder",          # caption fallback
    "unlock the door": "any_door_open",
    "go to the goal": "at_goal",
    "reach the goal": "at_goal",
}


### B8. Statistics (IQM + bootstrap CI, per Agarwal et al. 2021)

Interquartile mean + stratified bootstrap confidence intervals for per-seed scores. This is the reporting protocol the paper commits to — mean $\pm$ std is deliberately avoided because seed distributions in RL are typically skewed.


In [ ]:
"""Metrics + IQM + bootstrap CI per Agarwal, Schwarzer, Castro, Courville, Bellemare (2021).

Agarwal et al. "Deep Reinforcement Learning at the Edge of the Statistical Precipice",
NeurIPS 2021, arXiv:2108.13264.

Never report mean ± std here — the seed distribution in RL is skewed and mean±std
misleads. IQM (interquartile mean) + stratified bootstrap CI is the recommended
protocol.
"""
from __future__ import annotations

import numpy as np

def iqm(scores: np.ndarray) -> float:
    """Interquartile mean: drop the top 25% and bottom 25%, mean the rest.

    For n < 4 falls back to the plain mean (single-quartile case is undefined).
    """
    s = np.asarray(scores, dtype=float).ravel()
    if s.size == 0:
        return float("nan")
    if s.size < 4:
        return float(s.mean())
    q1, q3 = np.quantile(s, [0.25, 0.75])
    keep = s[(s >= q1) & (s <= q3)]
    if keep.size == 0:
        return float(s.mean())
    return float(keep.mean())

def bootstrap_iqm_ci(scores: np.ndarray,
                     n_boot: int = 2000,
                     ci: float = 0.95,
                     rng: np.random.Generator | None = None) -> tuple[float, float]:
    """Stratified nonparametric bootstrap CI on IQM.

    Returns (lo, hi) at the requested confidence level (default 95%).
    """
    s = np.asarray(scores, dtype=float).ravel()
    if s.size == 0:
        return float("nan"), float("nan")
    if rng is None:
        rng = np.random.default_rng(0)
    boots = np.empty(n_boot, dtype=float)
    n = s.size
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = iqm(s[idx])
    alpha = (1.0 - ci) / 2.0
    return float(np.quantile(boots, alpha)), float(np.quantile(boots, 1.0 - alpha))

def steps_to_threshold(episodes: np.ndarray,
                       success_rates: np.ndarray,
                       threshold: float = 0.5) -> float:
    """Sample-efficiency proxy: first episode index at which a rolling success
    rate crosses the threshold. Returns inf if never crossed.
    """
    eps = np.asarray(episodes, dtype=float).ravel()
    sr = np.asarray(success_rates, dtype=float).ravel()
    if eps.size == 0 or sr.size == 0:
        return float("inf")
    for e, r in zip(eps, sr):
        if r >= threshold:
            return float(e)
    return float("inf")

def rolling_success_rate(success_flags: np.ndarray, window: int = 100) -> np.ndarray:
    """Rolling mean of the last `window` successes at each episode index."""
    s = np.asarray(success_flags, dtype=float).ravel()
    if s.size == 0:
        return s
    out = np.empty_like(s)
    csum = np.cumsum(s)
    for i in range(s.size):
        lo = max(0, i - window + 1)
        out[i] = (csum[i] - (csum[lo - 1] if lo > 0 else 0.0)) / (i - lo + 1)
    return out

def summarize_run(returns: np.ndarray,
                  successes: np.ndarray,
                  n_boot: int = 2000) -> dict[str, float]:
    """Per-seed summary. Used before aggregating across seeds."""
    return {
        "final_return_mean": float(np.mean(returns)) if returns.size else float("nan"),
        "success_rate": float(np.mean(successes)) if successes.size else float("nan"),
        "n_episodes": int(returns.size),
    }

def aggregate_across_seeds(per_seed: list[dict[str, float]],
                           metric: str,
                           n_boot: int = 2000) -> dict[str, float]:
    """IQM + 95% bootstrap CI across seeds for a given per-seed scalar."""
    xs = np.array([r[metric] for r in per_seed if not np.isnan(r.get(metric, np.nan))])
    if xs.size == 0:
        return {"iqm": float("nan"), "lo": float("nan"), "hi": float("nan"), "n": 0}
    m = iqm(xs)
    lo, hi = bootstrap_iqm_ci(xs, n_boot=n_boot)
    return {"iqm": m, "lo": lo, "hi": hi, "n": int(xs.size)}


### B9. Reporting (per-seed CSV → Markdown table)

Reads a directory of per-seed CSV logs and produces a Markdown table with IQM $\pm$ 95 % bootstrap CI columns.


In [ ]:
"""Read a directory of per-seed CSV logs → emit a Markdown table with IQM ± 95% CI.

Usage:
    python eval/report.py --logs runs/ --out results/table.md
    python eval/report.py --logs pilot/logs/ --out pilot/pilot_output.md \
                          --group-by config --window 100
"""
from __future__ import annotations

import argparse
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd

# (defined in an earlier cell)

def load_logs(logs_glob: str) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(logs_glob, "**", "*.csv"), recursive=True) +
                   glob.glob(os.path.join(logs_glob, "*.csv")))
    files = list(dict.fromkeys(files))  # dedupe, keep order
    if not files:
        raise FileNotFoundError(f"No CSVs found under {logs_glob}")
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
        except pd.errors.EmptyDataError:
            continue
        if df.empty:
            continue
        df["source"] = f
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

def per_seed_summary(df: pd.DataFrame, tail_frac: float = 0.25) -> pd.DataFrame:
    """For each (config, seed): success rate on the last tail_frac of episodes,
    mean return on last tail, steps to reach 50% rolling success, per-seed totals."""
    rows = []
    for (cfg, seed), g in df.groupby(["config", "seed"]):
        g = g.sort_values("episode")
        n = len(g)
        if n == 0:
            continue
        tail = max(1, int(n * tail_frac))
        tail_g = g.tail(tail)
        rolling = rolling_success_rate(g["success"].to_numpy(), window=100)
        s2t = steps_to_threshold(g["step"].to_numpy(), rolling, threshold=0.5)
        rows.append({
            "config": cfg,
            "seed": int(seed),
            "n_episodes": int(n),
            "final_success_rate": float(tail_g["success"].mean()),
            "final_return": float(tail_g["return"].mean()),
            "steps_to_50pct": float(s2t),
            "total_steps": int(g["step"].max()),
            "total_llm_calls": int(g["llm_calls"].max()) if "llm_calls" in g else 0,
            "total_llm_tokens_out": int(g["llm_tokens_out"].max()) if "llm_tokens_out" in g else 0,
            "total_llm_wallclock_sec": float(g["llm_wallclock_sec"].max()) if "llm_wallclock_sec" in g else 0.0,
        })
    return pd.DataFrame(rows)

def aggregate_table(per_seed: pd.DataFrame,
                    metrics: list[str] = ("final_success_rate", "final_return",
                                          "steps_to_50pct"),
                    n_boot: int = 2000) -> pd.DataFrame:
    """IQM + 95% bootstrap CI per config per metric."""
    rng = np.random.default_rng(0)
    rows = []
    for cfg, g in per_seed.groupby("config"):
        row = {"config": cfg, "n_seeds": int(g.shape[0])}
        for m in metrics:
            vals = g[m].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                row[f"{m}_iqm"] = float("nan")
                row[f"{m}_lo"] = float("nan")
                row[f"{m}_hi"] = float("nan")
                continue
            row[f"{m}_iqm"] = iqm(vals)
            lo, hi = bootstrap_iqm_ci(vals, n_boot=n_boot, rng=rng)
            row[f"{m}_lo"] = lo
            row[f"{m}_hi"] = hi
        rows.append(row)
    return pd.DataFrame(rows)

def format_markdown(agg: pd.DataFrame,
                    per_seed: pd.DataFrame,
                    title: str = "Results (IQM ± 95% bootstrap CI)") -> str:
    """Emit a compact Markdown table with headline metrics + a compute footer."""
    def fmt(row, m, pct=False, digits=3):
        v, lo, hi = row[f"{m}_iqm"], row[f"{m}_lo"], row[f"{m}_hi"]
        if not np.isfinite(v):
            return "—"
        if pct:
            return f"{v*100:.1f}% [{lo*100:.1f}, {hi*100:.1f}]"
        if not np.isfinite(lo) or not np.isfinite(hi):
            return f"{v:.{digits}f}"
        if m == "steps_to_50pct":
            if not np.isfinite(v):
                return "never"
            return f"{int(v):,}"
        return f"{v:.{digits}f} [{lo:.{digits}f}, {hi:.{digits}f}]"

    lines = [f"### {title}", ""]
    lines.append("| Config | n seeds | Success rate | Final return | Steps to 50% |")
    lines.append("|---|---:|---|---|---|")
    for _, row in agg.iterrows():
        lines.append(
            f"| `{row['config']}` | {int(row['n_seeds'])} "
            f"| {fmt(row, 'final_success_rate', pct=True)} "
            f"| {fmt(row, 'final_return')} "
            f"| {fmt(row, 'steps_to_50pct')} |"
        )

    # Compute footer
    total_wc = per_seed["total_llm_wallclock_sec"].sum() if "total_llm_wallclock_sec" in per_seed else 0.0
    total_calls = per_seed["total_llm_calls"].sum() if "total_llm_calls" in per_seed else 0
    total_tokens = per_seed["total_llm_tokens_out"].sum() if "total_llm_tokens_out" in per_seed else 0
    lines += [
        "",
        f"*LLM inference budget across all runs: {int(total_calls):,} calls, "
        f"{int(total_tokens):,} output tokens, "
        f"{total_wc/60:.1f} minutes wallclock.*",
    ]
    return "\n".join(lines) + "\n"

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--logs", required=True, help="Directory containing per-seed CSVs")
    p.add_argument("--out", required=True, help="Markdown output path")
    p.add_argument("--title", default="Results (IQM ± 95% bootstrap CI)")
    p.add_argument("--tail-frac", type=float, default=0.25)
    p.add_argument("--n-boot", type=int, default=2000)
    args = p.parse_args()

    df = load_logs(args.logs)
    per_seed = per_seed_summary(df, tail_frac=args.tail_frac)
    agg = aggregate_table(per_seed, n_boot=args.n_boot)
    md = format_markdown(agg, per_seed, title=args.title)

    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(md, encoding="utf-8")
    print(f"[report] wrote {args.out}")
    print(md)

if __name__ == "__main__":
    main()


## C. Demo: Reproduce Table 1 (Proposition 1 numerical verification)

Enumerates policies on a 3-state MDP and checks that four $\Phi$ configurations all preserve the base optimal policy.

**Runtime:** about 5 seconds on CPU. No LLM required.


In [ ]:
"""Numerical verification of Proposition 1 (potential-based shaping is policy-invariant).

Enumerates all deterministic policies on a small MDP and verifies that the optimal
policy of the shaped MDP equals the optimal policy of the base MDP for four Phi
configurations (zero, constant, sign-flip, adversarial-large).

Output: a Markdown table drop-in for paper §4.5.1.

Usage:
    python verify/prop1_numerical.py --out verify/prop1_output.md
"""
from __future__ import annotations

import argparse
import itertools
from pathlib import Path

import numpy as np

# 3-state, 2-action deterministic MDP
#   States: 0 (start), 1 (intermediate), 2 (terminal goal)
#   Actions: 0 (stay/back), 1 (forward)
TRANS = {
    (0, 0): 0, (0, 1): 1,
    (1, 0): 0, (1, 1): 2,
    (2, 0): 2, (2, 1): 2,
}
REWARD = {
    (0, 0, 0): -0.1,
    (0, 1, 1): 0.0,
    (1, 0, 0): -0.1,
    (1, 1, 2): 1.0,
    (2, 0, 2): 0.0,
    (2, 1, 2): 0.0,
}
GAMMA = 0.9
HORIZON = 200

PHI_CONFIGS = {
    "$\\Phi \\equiv 0$":        {0: 0.0, 1: 0.0, 2: 0.0},
    "$\\Phi \\equiv 1$":        {0: 1.0, 1: 1.0, 2: 1.0},
    "sign-flip":                {0: -0.5, 1: 0.5, 2: 0.0},
    "adversarial large":       {0: 10.0, 1: -10.0, 2: 5.0},
}

def evaluate_policy(policy: dict[int, int],
                    phi_fn=None,
                    gamma: float = GAMMA,
                    horizon: int = HORIZON) -> float:
    s = 0
    total = 0.0
    disc = 1.0
    for _ in range(horizon):
        a = policy[s]
        s_next = TRANS[(s, a)]
        r = REWARD[(s, a, s_next)]
        if phi_fn is not None:
            r = r + gamma * phi_fn(s_next) - phi_fn(s)
        total += disc * r
        disc *= gamma
        if s_next == 2:
            break
        s = s_next
    return total

def enumerate_policies():
    for a0, a1 in itertools.product([0, 1], repeat=2):
        yield {0: a0, 1: a1, 2: 0}

def find_optimal(phi_fn=None) -> tuple[tuple[int, int], float]:
    best = None
    best_val = -np.inf
    for pol in enumerate_policies():
        v = evaluate_policy(pol, phi_fn=phi_fn)
        if v > best_val:
            best_val = v
            best = (pol[0], pol[1])
    return best, best_val

def policy_str(pol: tuple[int, int]) -> str:
    """(a_at_state_0, a_at_state_1) -> compact human string."""
    names = {0: "stay", 1: "forward"}
    return f"[s0: {names[pol[0]]}, s1: {names[pol[1]]}]"

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="verify/prop1_output.md")
    args = ap.parse_args()

    base_optimal, base_V = find_optimal(phi_fn=None)

    rows = []
    all_match = True
    for name, phi_map in PHI_CONFIGS.items():
        shaped_optimal, shaped_V = find_optimal(phi_fn=lambda s, m=phi_map: m[s])
        match = shaped_optimal == base_optimal
        all_match = all_match and match
        rows.append({
            "phi": name,
            "base_V": base_V,
            "shaped_V": shaped_V,
            "shaped_optimal": policy_str(shaped_optimal),
            "match": "yes" if match else "**no**",
        })

    lines = [
        "### Numerical verification of Proposition 1",
        "",
        f"3-state deterministic MDP; $\\gamma = {GAMMA}$; 4 deterministic policies enumerated. "
        f"Base optimal policy: {policy_str(base_optimal)} with $V^\\ast(s_0) = {base_V:.4f}$.",
        "",
        "| $\\Phi$ configuration | Base $V^\\ast(s_0)$ | Shaped $\\tilde V^\\ast(s_0)$ | Shaped optimal policy | Matches base? |",
        "|---|---:|---:|---|---|",
    ]
    for r in rows:
        lines.append(
            f"| {r['phi']} | {r['base_V']:.4f} | {r['shaped_V']:.4f} | "
            f"{r['shaped_optimal']} | {r['match']} |"
        )
    lines += [
        "",
        f"All four $\\Phi$ configurations preserve the base optimal policy "
        f"({'confirmed' if all_match else '**violation detected**'}), matching the theoretical guarantee. "
        f"The shaped value $\\tilde V^\\ast$ differs from $V^\\ast$ by the potential offset "
        f"$-\\Phi(s_0)$ plus a vanishing telescope tail, as expected.",
    ]

    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    print("\n".join(lines))
    print()
    print(f"[verify] wrote {out_path}")
    if not all_match:
        raise SystemExit(1)

if __name__ == "__main__":
    main()


**Run the demo:**


In [ ]:
# Run the Prop 1 verification and print the Markdown table.
import sys as _sys, tempfile as _tempfile
from pathlib import Path as _Path
_out_path = _Path(_tempfile.gettempdir()) / "prop1_output.md"
_argv_backup = _sys.argv[:]
_sys.argv = ["prop1_numerical.py", "--out", str(_out_path)]
try:
    main()
finally:
    _sys.argv = _argv_backup


## D. Demo: Reproduce Table 2 (planner-in-isolation audit)

Runs the P1 planning prompt through the LLM backend selected in Section A on 20 hand-curated MiniGrid tasks and scores the outputs against ground truth.

**Runtime:**
- Ollama with `qwen2.5:14b` (CPU): about 30 min.
- Groq with `llama-3.1-8b-instant`: about 2 min.
- Mock backend: instant (no real LLM).

**Reproducibility note.** The paper's Table 2 values were produced with Ollama + Qwen-2.5:14b. Reproducing with Groq/Llama or any other LLM exercises the same framework but gives different numbers. Same shape, different values.


### D1. The 20 audit tasks (verbatim from `code/audit/tasks.json`)


In [ ]:
AUDIT_TASKS = {
  "version": "1.0",
  "description": "20 MiniGrid tasks with ground-truth subgoal decompositions. Hand-curated. Used by audit/plan_quality.py to score LLM plans in isolation (no RL training).",
  "tasks": [
    {
      "id": "empty_goto_goal",
      "task": "Reach the green goal.",
      "state": "8x8 empty room. Agent at (1, 1) facing right. Carrying: nothing. Objects: green goal at (6, 6).",
      "ground_truth": ["go to the goal"]
    },
    {
      "id": "doorkey_yellow_6x6",
      "task": "Use the yellow key to unlock the yellow door and reach the green goal.",
      "state": "6x6 room split by a wall. Agent at (1, 4) facing right. Carrying: nothing. Objects: yellow key at (2, 4), yellow door at (3, 2) closed and locked, green goal at (5, 4).",
      "ground_truth": ["go to the key", "pick up the key", "go to the door", "unlock the door", "go to the goal"]
    },
    {
      "id": "doorkey_red_8x8",
      "task": "Open the red door with the red key and reach the goal on the other side.",
      "state": "8x8 room split by a vertical wall. Agent at (2, 5) facing up. Carrying: nothing. Objects: red key at (1, 6), red door at (4, 3) closed and locked, green goal at (7, 5).",
      "ground_truth": ["go to the key", "pick up the key", "go to the door", "unlock the door", "go to the goal"]
    },
    {
      "id": "fetch_blue_ball",
      "task": "Pick up the blue ball.",
      "state": "5x5 room. Agent at (2, 2) facing down. Carrying: nothing. Objects: blue ball at (1, 3), red ball at (4, 1), green key at (3, 4).",
      "ground_truth": ["go to the blue ball", "pick up the blue ball"]
    },
    {
      "id": "fetch_green_key",
      "task": "Pick up the green key.",
      "state": "5x5 room. Agent at (2, 2) facing down. Carrying: nothing. Objects: blue ball at (1, 3), red ball at (4, 1), green key at (3, 4).",
      "ground_truth": ["go to the green key", "pick up the green key"]
    },
    {
      "id": "unlock_yellow_door",
      "task": "Unlock the yellow door.",
      "state": "5x5 room. Agent at (1, 1) facing right. Carrying: nothing. Objects: yellow key at (3, 1), yellow door at (4, 3) closed and locked.",
      "ground_truth": ["go to the key", "pick up the key", "go to the door", "unlock the door"]
    },
    {
      "id": "unlockpickup_box",
      "task": "Pick up the blue box in the next room. The door is locked.",
      "state": "6x6 room. Agent at (1, 3) facing right. Carrying: nothing. Objects: yellow key at (2, 4), yellow door at (3, 3) closed and locked, blue box at (5, 3).",
      "ground_truth": ["go to the key", "pick up the key", "go to the door", "unlock the door", "go to the box", "pick up the box"]
    },
    {
      "id": "keycorridor_ball",
      "task": "Pick up the blue ball. You may need a key to unlock a door.",
      "state": "3-room corridor. Agent at (1, 4) facing right. Carrying: nothing. Objects: blue key at (2, 2), red door at (5, 4) closed and locked, blue ball at (8, 4).",
      "ground_truth": ["go to the key", "pick up the key", "go to the door", "unlock the door", "go to the ball", "pick up the ball"]
    },
    {
      "id": "multiroom_n2",
      "task": "Reach the goal by passing through the door between the two rooms.",
      "state": "Two adjoining rooms. Agent at (1, 2) in the left room facing right. Carrying: nothing. Objects: door at (4, 3) closed, green goal at (8, 3) in the right room.",
      "ground_truth": ["go to the door", "open the door", "go to the goal"]
    },
    {
      "id": "multiroom_n3",
      "task": "Reach the goal by passing through two doors across three rooms.",
      "state": "Three adjoining rooms. Agent at (1, 2) in the leftmost room facing right. Carrying: nothing. Objects: door at (4, 3) closed, door at (8, 3) closed, green goal at (11, 3).",
      "ground_truth": ["go to the first door", "open the first door", "go to the second door", "open the second door", "go to the goal"]
    },
    {
      "id": "putnext_ball_key",
      "task": "Put the blue ball next to the yellow key.",
      "state": "6x6 room. Agent at (2, 2) facing right. Carrying: nothing. Objects: blue ball at (3, 4), yellow key at (5, 1).",
      "ground_truth": ["go to the blue ball", "pick up the blue ball", "go to the yellow key", "drop the blue ball next to the key"]
    },
    {
      "id": "goto_red_box",
      "task": "Go to the red box.",
      "state": "6x6 room. Agent at (1, 1) facing right. Carrying: nothing. Objects: red box at (4, 5), yellow key at (2, 3), blue ball at (5, 2).",
      "ground_truth": ["go to the red box"]
    },
    {
      "id": "doorkey_conjoined",
      "task": "Reach the green goal on the far side of the locked yellow door.",
      "state": "6x6 room split by a wall. Agent at (3, 5) facing up. Carrying: nothing. Objects: yellow key at (1, 4), yellow door at (3, 2) closed and locked, green goal at (4, 1).",
      "ground_truth": ["go to the key", "pick up the key", "go to the door", "unlock the door", "go to the goal"]
    },
    {
      "id": "carrying_key_open_door",
      "task": "Open the yellow door. You are already carrying the yellow key.",
      "state": "5x5 room. Agent at (2, 3) facing right. Carrying: yellow key. Objects: yellow door at (4, 3) closed and locked.",
      "ground_truth": ["go to the door", "unlock the door"]
    },
    {
      "id": "carrying_key_reach_goal",
      "task": "Reach the green goal, unlocking the door if needed. You are already carrying the yellow key.",
      "state": "6x6 room. Agent at (1, 3) facing right. Carrying: yellow key. Objects: yellow door at (3, 3) closed and locked, green goal at (5, 3).",
      "ground_truth": ["go to the door", "unlock the door", "go to the goal"]
    },
    {
      "id": "door_already_open",
      "task": "Reach the green goal.",
      "state": "6x6 room split by a wall. Agent at (1, 3) facing right. Carrying: nothing. Objects: yellow door at (3, 3) open, green goal at (5, 3).",
      "ground_truth": ["go to the goal"]
    },
    {
      "id": "wrong_color_key_present",
      "task": "Unlock the red door and reach the goal.",
      "state": "6x6 room split by a wall. Agent at (1, 3) facing right. Carrying: nothing. Objects: red key at (2, 4), yellow key at (2, 2), red door at (3, 3) closed and locked, green goal at (5, 3).",
      "ground_truth": ["go to the red key", "pick up the red key", "go to the door", "unlock the door", "go to the goal"]
    },
    {
      "id": "fetch_specify_color",
      "task": "Pick up the yellow key.",
      "state": "5x5 room. Agent at (2, 2) facing right. Carrying: nothing. Objects: yellow key at (3, 1), red key at (1, 3), yellow ball at (4, 4).",
      "ground_truth": ["go to the yellow key", "pick up the yellow key"]
    },
    {
      "id": "goto_facing_wrong_way",
      "task": "Reach the green goal.",
      "state": "8x8 empty room. Agent at (6, 6) facing left. Carrying: nothing. Objects: green goal at (1, 1).",
      "ground_truth": ["go to the goal"]
    },
    {
      "id": "obstacle_around_object",
      "task": "Pick up the yellow key surrounded by walls; find the opening.",
      "state": "8x8 room with an interior 3x3 alcove. Agent at (1, 1) facing right. Carrying: nothing. Objects: yellow key at (4, 4) inside the alcove; alcove entrance at (5, 4).",
      "ground_truth": ["navigate to the alcove entrance", "go to the key", "pick up the key"]
    }
  ]
}


### D2. Audit runner


In [ ]:
"""Planner-in-isolation audit: run the P1 prompt through Ollama on 20 MiniGrid tasks
with known ground-truth decompositions, and score plan quality.

Metrics (per task):
  parse_success   : bool         — did the LLM output parse as >=1 subgoal?
  plan_length     : int
  coverage        : float in [0,1] — fraction of ground-truth subgoals matched
                                     (in order) by some subgoal in the LLM plan
  extraneous      : int         — LLM subgoals with no ground-truth match
  tokens_out      : int
  wallclock_sec   : float
  parseable_length: int (=plan_length if parsed)

Aggregate report: parse rate, mean plan length, mean coverage, mean extraneous,
mean/median tokens, mean/median wall clock.

Every LLM raw output is saved to audit/raw_plans/{task_id}.txt for audit.

Usage:
    python audit/plan_quality.py --model qwen2.5:14b --tasks audit/tasks.json \
                                 --out audit/plan_quality_output.md
"""
from __future__ import annotations

import argparse
import json
import re
import statistics
import sys
import time
from pathlib import Path

# (defined in an earlier cell)

# --------------------------------------------------------------------------
# Fuzzy matching between LLM subgoals and ground-truth subgoals
# --------------------------------------------------------------------------

_STOPWORDS = {
    "the", "a", "an", "to", "of", "at", "in", "on", "up", "and", "or",
    "then", "next", "after", "before", "with",
}

def _tokens(s: str) -> set[str]:
    ws = re.findall(r"\w+", s.lower())
    return {w for w in ws if w not in _STOPWORDS and len(w) >= 3}

def subgoal_match(pred: str, truth: str, threshold: float = 0.5) -> bool:
    """Recall-oriented match: fraction of ground-truth content tokens present in pred
    is at least `threshold`. Motivation: LLMs often produce verbose or synonymous
    subgoals (e.g. 'navigate to the yellow key' for GT 'go to the key'); a symmetric
    Jaccard penalises them unfairly. Recall of the GT content is the right signal.
    """
    a, b = _tokens(pred), _tokens(truth)
    if not b:
        return False
    if not a:
        return False
    covered = len(a & b) / len(b)
    return covered >= threshold

def score_plan(plan: list[str], ground_truth: list[str]) -> dict:
    """Compute coverage (in-order matches of ground truth) + extraneous count."""
    if not plan:
        return {"coverage": 0.0, "extraneous": 0, "matched_gt": 0}
    matched_gt_indices: list[int] = []
    used_pred = set()
    j = 0  # pointer into ground_truth
    for i, pred in enumerate(plan):
        if j >= len(ground_truth):
            break
        if subgoal_match(pred, ground_truth[j]):
            matched_gt_indices.append(j)
            used_pred.add(i)
            j += 1
    # extraneous = pred subgoals that never matched any GT (in order or otherwise)
    extraneous = 0
    for i, pred in enumerate(plan):
        if i in used_pred:
            continue
        if not any(subgoal_match(pred, g) for g in ground_truth):
            extraneous += 1
    coverage = len(matched_gt_indices) / len(ground_truth) if ground_truth else 0.0
    return {"coverage": coverage, "extraneous": extraneous,
            "matched_gt": len(matched_gt_indices)}

# --------------------------------------------------------------------------
# Audit runner
# --------------------------------------------------------------------------

def run_audit(tasks: list[dict], planner: OllamaPlanner,
              raw_dir: Path, verbose: bool = True) -> list[dict]:
    results = []
    raw_dir.mkdir(parents=True, exist_ok=True)
    for i, t in enumerate(tasks):
        t0 = time.perf_counter()
        toks_in_before = planner.stats.tokens_in
        toks_out_before = planner.stats.tokens_out
        calls_before = planner.stats.calls

        # Call the same planning path used at training time
        try:
            plan = planner.plan(task=t["task"], state_caption=t["state"])
            err = None
        except Exception as e:
            plan = []
            err = str(e)

        wallclock = time.perf_counter() - t0
        toks_in = planner.stats.tokens_in - toks_in_before
        toks_out = planner.stats.tokens_out - toks_out_before
        n_calls = planner.stats.calls - calls_before

        parsed_ok = len(plan) > 0
        sc = score_plan(plan, t["ground_truth"])

        # Save raw text: we re-run raw to capture the LLM output verbatim
        raw_text = "\n".join(plan) if plan else "(no parseable output)"
        (raw_dir / f"{t['id']}.txt").write_text(
            f"# task: {t['task']}\n"
            f"# state: {t['state']}\n"
            f"# ground_truth: {t['ground_truth']}\n\n"
            f"{raw_text}\n",
            encoding="utf-8",
        )

        row = {
            "id": t["id"],
            "task": t["task"],
            "parse_success": parsed_ok,
            "plan_length": len(plan),
            "ground_truth_length": len(t["ground_truth"]),
            "coverage": sc["coverage"],
            "extraneous": sc["extraneous"],
            "matched_gt": sc["matched_gt"],
            "tokens_out": toks_out,
            "tokens_in": toks_in,
            "wallclock_sec": wallclock,
            "n_calls": n_calls,
            "error": err,
        }
        results.append(row)

        if verbose:
            print(f"[{i+1:2}/{len(tasks)}] {t['id']:<32} "
                  f"parse={'y' if parsed_ok else 'n'} "
                  f"len={len(plan):>2} "
                  f"cov={sc['coverage']*100:>5.1f}% "
                  f"extra={sc['extraneous']:>2} "
                  f"toks_out={toks_out:>4} "
                  f"wc={wallclock:>5.1f}s")
    return results

def aggregate(results: list[dict]) -> dict:
    parsed = [r for r in results if r["parse_success"]]
    n_total = len(results)
    n_parsed = len(parsed)

    def stat(xs, fn=statistics.mean, fallback=float("nan")):
        return fn(xs) if xs else fallback

    return {
        "n_total": n_total,
        "n_parsed": n_parsed,
        "parse_rate": (n_parsed / n_total) if n_total else 0.0,
        "mean_plan_length": stat([r["plan_length"] for r in parsed]),
        "median_plan_length": stat([r["plan_length"] for r in parsed], fn=statistics.median),
        "mean_coverage": stat([r["coverage"] for r in results]),
        "mean_extraneous": stat([r["extraneous"] for r in results]),
        "mean_tokens_out": stat([r["tokens_out"] for r in results]),
        "median_tokens_out": stat([r["tokens_out"] for r in results], fn=statistics.median),
        "mean_wallclock_sec": stat([r["wallclock_sec"] for r in results]),
        "median_wallclock_sec": stat([r["wallclock_sec"] for r in results], fn=statistics.median),
        "total_wallclock_sec": sum(r["wallclock_sec"] for r in results),
    }

def format_markdown(agg: dict, model: str, n_tasks: int) -> str:
    lines = [
        "### Planner-in-isolation audit (Qwen-2.5:14b via Ollama)",
        "",
        f"Model: `{model}`. Prompt template: P1 (Appendix~\\ref{{app:prompts}}). "
        f"Task set: {n_tasks} hand-curated MiniGrid tasks with ground-truth "
        f"subgoal decompositions (see \\texttt{{code/audit/tasks.json}}). "
        f"Grader: an LLM subgoal matches a ground-truth subgoal when at least "
        f"$50\\%$ of the ground-truth's content tokens (non-stopword, length $\\geq 3$) "
        f"appear in the LLM subgoal. Coverage is the fraction of ground-truth subgoals "
        f"matched in order by the LLM plan; extraneous is the number of LLM subgoals "
        f"with no ground-truth match. No RL training in this study.",
        "",
        "| Metric | Value |",
        "|---|---|",
        f"| Parse rate (fraction of outputs $\\geq 1$ subgoal) | "
        f"{agg['parse_rate']*100:.1f}% ({agg['n_parsed']}/{agg['n_total']}) |",
        f"| Mean plan length (parsed only) | {agg['mean_plan_length']:.2f} subgoals |",
        f"| Median plan length | {int(agg['median_plan_length'])} subgoals |",
        f"| Mean ground-truth coverage | {agg['mean_coverage']*100:.1f}% |",
        f"| Mean extraneous subgoals per plan | {agg['mean_extraneous']:.2f} |",
        f"| Median tokens out per plan | {int(agg['median_tokens_out'])} |",
        f"| Median wall clock per plan | {agg['median_wallclock_sec']:.2f}\\,s |",
        f"| Total wall clock ({agg['n_total']} plans) | "
        f"{agg['total_wallclock_sec']/60:.1f}\\,min |",
        "",
        f"Reading: the LLM parses cleanly on {agg['parse_rate']*100:.0f}% of tasks and covers "
        f"{agg['mean_coverage']*100:.0f}% of ground-truth subgoals on average. Extraneous subgoals "
        f"(LLM subgoals with no ground-truth match) average {agg['mean_extraneous']:.1f} per plan. "
        f"This is the planner in isolation --- it does not measure the hybrid's RL performance, "
        f"which requires the training runs of the \\S\\ref{{sec:experiments}} protocol.",
    ]
    return "\n".join(lines) + "\n"

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tasks", default="audit/tasks.json")
    ap.add_argument("--model", default="qwen2.5:14b")
    ap.add_argument("--out", default="audit/plan_quality_output.md")
    ap.add_argument("--raw-dir", default="audit/raw_plans")
    ap.add_argument("--host", default=None, help="Ollama host (default: http://localhost:11434)")
    args = ap.parse_args()

    tasks_data = json.loads(Path(args.tasks).read_text(encoding="utf-8"))
    tasks = tasks_data["tasks"]

    planner = OllamaPlanner(model=args.model, host=args.host)
    print(f"[audit] running P1 on {len(tasks)} tasks with model={args.model}")

    results = run_audit(tasks, planner, raw_dir=Path(args.raw_dir))
    agg = aggregate(results)

    print()
    print(f"[audit] parse_rate={agg['parse_rate']*100:.1f}%  "
          f"coverage={agg['mean_coverage']*100:.1f}%  "
          f"extra={agg['mean_extraneous']:.2f}  "
          f"median_wc={agg['median_wallclock_sec']:.2f}s")

    md = format_markdown(agg, model=args.model, n_tasks=len(tasks))
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(md, encoding="utf-8")
    print(f"[audit] wrote {args.out}")

    # Also dump per-task JSON for the appendix / audit trail
    per_task_path = Path(args.out).with_suffix(".json")
    per_task_path.write_text(json.dumps(
        {"model": args.model, "aggregate": agg, "per_task": results},
        indent=2), encoding="utf-8")
    print(f"[audit] wrote {per_task_path}")

if __name__ == "__main__":
    main()


**Run the demo** (skips if Ollama is not available):


In [ ]:
# Run the audit using whichever LLM backend was auto-selected in Section A.
if LLM_BACKEND == "mock":
    print("Skipping audit: no real LLM available (LLM_BACKEND == 'mock').")
    print("Set up Ollama or Groq (see Section A) and re-run.")
else:
    import json as _json, tempfile as _tempfile, sys as _sys, time as _time
    from pathlib import Path as _Path

    # Build the appropriate planner instance directly (bypass CLI-argv path
    # so we can pick between Ollama and Groq at runtime).
    if LLM_BACKEND == "ollama":
        _planner = OllamaPlanner(model="qwen2.5:14b")
        _model_label = "qwen2.5:14b (Ollama)"
    else:  # groq
        _planner = GroqPlanner(model="llama-3.1-8b-instant")
        _model_label = "llama-3.1-8b-instant (Groq)"

    _tmp = _Path(_tempfile.mkdtemp())
    _raw_dir = _tmp / "raw_plans"
    _raw_dir.mkdir()
    _out_md = _tmp / "plan_quality_output.md"

    print(f"Running audit against {_model_label} on {len(AUDIT_TASKS['tasks'])} tasks...")
    _t0 = _time.time()
    _results = run_audit(AUDIT_TASKS["tasks"], _planner, raw_dir=_raw_dir, verbose=True)
    _agg = aggregate(_results)
    _md = format_markdown(_agg, model=_model_label, n_tasks=len(_results))
    _out_md.write_text(_md, encoding="utf-8")
    print(f"\nTotal wall clock: {(_time.time()-_t0)/60:.1f} min")
    print("---")
    print("Generated table:")
    print(_md)


## E. Demo: Reproduce Table 3 (pipeline validation pilot)

Runs PPO baseline vs. Hybrid on MiniGrid-DoorKey-6x6 for 3 seeds $\times$ 30k steps each, then aggregates IQM + bootstrap CI.

**Runtime:**
- PPO baseline: about 5 min per seed $\times$ 3 seeds = 15 min total (no LLM).
- Hybrid with Ollama + `qwen2.5:14b`: 30 to 90 min per seed depending on CPU speed.
- Hybrid with Groq + `llama-3.1-8b-instant`: 5 to 10 min per seed (API latency dominates).

**Reproducibility note.** The paper's Table 3 values were produced with Ollama + Qwen-2.5:14b. Groq/Llama exercises the same pipeline with different numbers. PPO baseline numbers should be roughly reproducible across backends (no LLM involved).

If you just want to see the analysis machinery without running training, skip the training cell. The analysis cell reads whatever CSVs are present and reports on those.


### E1. Training loop


In [ ]:
"""Entrypoint. Wire an env + planner + shaper + scheduler + replan + PPO from a YAML
config and run for --steps environment steps. Log per-episode CSV to --out.

Usage:
    python train.py --config baselines/configs/ppo.yaml \
                    --env MiniGrid-DoorKey-6x6-v0 --steps 200000 --seed 0
    python train.py --config baselines/configs/hybrid.yaml --steps 500 --planner mock --dry-run
"""
from __future__ import annotations

import argparse
import csv
import os
import random
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import yaml

# --------------------------------------------------------------------------
# Env construction
# --------------------------------------------------------------------------

def _apply_env_wraps(env, wraps: list[str]):
    """Apply MiniGrid-family wrappers by name."""
    for w in wraps:
        if w == "flat_obs":
            from minigrid.wrappers import FlatObsWrapper
            env = FlatObsWrapper(env)
        else:
            raise ValueError(f"Unknown env wrap: {w}")
    return env

def build_base_env(env_id: str, wraps: list[str], seed: int):
    import gymnasium as gym
    import minigrid  # noqa: F401  (side-effect: registers MiniGrid-* envs)
    env = gym.make(env_id)
    env = _apply_env_wraps(env, wraps)
    env.reset(seed=seed)
    return env

# --------------------------------------------------------------------------
# Planner / encoder factories
# --------------------------------------------------------------------------

def build_planner(cfg: dict[str, Any], override_backend: str | None = None):
    """Return a Planner. override_backend forces a specific backend regardless of cfg."""
    from hybrid.planner import OllamaPlanner, GroqPlanner, MockPlanner

    backend = override_backend or cfg.get("backend", "mock")
    if backend == "ollama":
        return OllamaPlanner(
            model=cfg.get("model", "qwen2.5:14b"),
            max_plan_length=cfg.get("max_plan_length", 12),
            max_subgoal_tokens=cfg.get("max_subgoal_tokens", 32),
            environment_name=cfg.get("environment_name", "MiniGrid"),
            temperature=cfg.get("temperature", 0.0),
        )
    if backend == "groq":
        return GroqPlanner(
            model=cfg.get("model", "llama-3.1-8b-instant"),
            max_plan_length=cfg.get("max_plan_length", 12),
            max_subgoal_tokens=cfg.get("max_subgoal_tokens", 32),
            environment_name=cfg.get("environment_name", "MiniGrid"),
            temperature=cfg.get("temperature", 0.0),
        )
    if backend == "mock":
        return MockPlanner(
            max_plan_length=cfg.get("max_plan_length", 12),
            max_subgoal_tokens=cfg.get("max_subgoal_tokens", 32),
            environment_name=cfg.get("environment_name", "MiniGrid"),
        )
    if backend == "random":
        return _build_random_planner(cfg)
    raise ValueError(f"Unknown planner backend: {backend}")

def _build_random_planner(cfg: dict[str, Any]):
    """Random-subgoal ablation. Samples subgoals from a fixed vocabulary."""
    from dataclasses import dataclass, field
    from hybrid.planner import PlannerStats
    vocab = list(cfg.get("vocabulary", ["explore", "move", "act"]))
    max_len = int(cfg.get("max_plan_length", 12))

    @dataclass
    class RandomPlanner:
        stats: PlannerStats = field(default_factory=PlannerStats)

        def plan(self, task, state_caption, failed_subgoal=None):
            self.stats.calls += 1
            n = random.randint(1, max_len)
            return [random.choice(vocab) for _ in range(n)]

        def score(self, subgoal, state_caption):
            self.stats.calls += 1
            return random.random()

        def completed(self, subgoal, state_caption):
            self.stats.calls += 1
            return random.random() < 0.1

    return RandomPlanner()

def build_encoder(cfg: dict[str, Any]):
    from hybrid.policy import STEncoder, MockEncoder
    backend = cfg.get("backend", "mock")
    if backend == "sentence_transformer":
        return STEncoder(model_name=cfg.get("model_name",
                                             "sentence-transformers/all-MiniLM-L6-v2"))
    if backend == "mock":
        return MockEncoder(embedding_dim=int(cfg.get("embedding_dim", 64)))
    raise ValueError(f"Unknown encoder backend: {backend}")

# --------------------------------------------------------------------------
# Hybrid env wrapper — the key integration point.
# --------------------------------------------------------------------------

def build_hybrid_env(base_env, cfg: dict, planner, encoder, seed: int,
                     csv_writer, run_metadata: dict):
    """Wrap base env with subgoal conditioning + shaping + scheduling + replan.
    Returns a gymnasium.Wrapper suitable for sb3 PPO.
    """
    import gymnasium as gym
    from hybrid.policy import make_subgoal_wrapper
    from hybrid.shaping import PotentialShaper
    from hybrid.schedule import SubgoalScheduler, EnvEventDone, LLMDone, MLPDone
    from hybrid.replan import ReplanTrigger
    from envs.minigrid_wrapper import caption_minigrid, extract_events, DEFAULT_SUBGOAL_TO_EVENT

    # Outer wrapper adds subgoal_embedding channel
    outer = make_subgoal_wrapper(base_env, encoder)

    shaper = PotentialShaper(
        planner=planner,
        gamma=cfg["shaping"].get("gamma", 0.99),
        magnitude=cfg["shaping"].get("magnitude", 1.0),
        enabled=cfg["shaping"].get("enabled", True),
    )

    done_kind = cfg["schedule"].get("done_oracle", "env_event")
    if done_kind == "env_event":
        oracle = EnvEventDone(event_extractor=lambda s: extract_events(base_env),
                              subgoal_to_event=dict(DEFAULT_SUBGOAL_TO_EVENT))
    elif done_kind == "llm":
        oracle = LLMDone(planner=planner)
    elif done_kind == "mlp":
        oracle = MLPDone()
    else:
        raise ValueError(f"Unknown done_oracle: {done_kind}")

    scheduler = SubgoalScheduler(plan=[], oracle=oracle)
    replanner = ReplanTrigger(
        planner=planner,
        period_H=cfg["replan"].get("period_H", 50),
        budget_B=cfg["replan"].get("budget_B", 25),
    )

    task_string = str(getattr(base_env.unwrapped, "mission", "reach the goal"))
    inner_env = base_env  # for captioner/events
    initial_plan_cache: dict[str, list[str]] = {}

    class HybridEnv(gym.Wrapper):
        def __init__(self, wrapped):
            super().__init__(wrapped)
            self._episode_return = 0.0
            self._episode_shaped_return = 0.0
            self._episode_steps = 0
            self._episode_idx = 0
            self._total_step = 0
            self._events = {"replan_periodic": 0, "replan_failure": 0,
                            "subgoal_advances": 0}
            self._prev_caption = ""

        def reset(self, **kwargs):
            obs, info = self.env.reset(**kwargs)
            self._episode_return = 0.0
            self._episode_shaped_return = 0.0
            self._episode_steps = 0
            self._events = {"replan_periodic": 0, "replan_failure": 0,
                            "subgoal_advances": 0}
            # Fresh task string per-episode (BabyAI resets randomize the mission)
            task = str(getattr(inner_env.unwrapped, "mission", task_string))
            caption = caption_minigrid(inner_env)
            # Cache plan by (task string). MiniGrid tasks have stable mission
            # strings for a fixed env id, so we avoid re-planning on every reset.
            # Periodic and failure replan triggers can still refresh the plan
            # mid-episode via ReplanTrigger.
            cached = initial_plan_cache.get(task)
            if cached is None:
                plan = planner.plan(task, caption)
                if not plan:
                    plan = ["reach the goal"]
                initial_plan_cache[task] = list(plan)
            else:
                plan = list(cached)
            scheduler.plan = plan
            scheduler.k = 0
            scheduler.b = 0
            # push initial subgoal into wrapper
            outer.set_subgoal(scheduler.active)
            self._prev_caption = caption
            return obs, info

        def step(self, action):
            obs, reward, terminated, truncated, info = self.env.step(action)
            self._episode_return += float(reward)
            self._episode_steps += 1
            self._total_step += 1

            next_caption = caption_minigrid(inner_env)
            active = scheduler.active

            # 1. Shape reward using potential at (prev, active) and (next, active)
            shaping_bonus = shaper.shape(self._prev_caption, next_caption, active) \
                            if active else 0.0
            shaped_reward = float(reward) + float(shaping_bonus)
            self._episode_shaped_return += shaped_reward

            # 2. Advance scheduler (checks Done oracle)
            diag = scheduler.step(inner_env, inner_env, next_caption)
            if diag["advanced"]:
                self._events["subgoal_advances"] += 1
                outer.set_subgoal(scheduler.active)

            # 3. Check replan triggers
            fired, reason = replanner.maybe_replan(
                self._total_step, task_string, next_caption, scheduler,
            )
            if fired:
                self._events[f"replan_{reason}"] += 1
                outer.set_subgoal(scheduler.active)

            self._prev_caption = next_caption

            # Log episode when done
            if terminated or truncated:
                self._episode_idx += 1
                success = bool(reward > 0)  # MiniGrid: only-terminal +ve reward
                if csv_writer is not None:
                    csv_writer.writerow({
                        "seed": run_metadata["seed"],
                        "config": run_metadata["config_name"],
                        "step": self._total_step,
                        "episode": self._episode_idx,
                        "return": round(self._episode_return, 6),
                        "shaped_return": round(self._episode_shaped_return, 6),
                        "success": int(success),
                        "steps": self._episode_steps,
                        "subgoal_advances": self._events["subgoal_advances"],
                        "replan_periodic": self._events["replan_periodic"],
                        "replan_failure": self._events["replan_failure"],
                        "llm_calls": planner.stats.calls,
                        "llm_tokens_in": planner.stats.tokens_in,
                        "llm_tokens_out": planner.stats.tokens_out,
                        "llm_wallclock_sec": round(planner.stats.wallclock_sec, 3),
                    })

            return obs, shaped_reward, terminated, truncated, info

    return HybridEnv(outer)

# --------------------------------------------------------------------------
# Pure-PPO env — no LLM, no subgoal channel, no shaping
# --------------------------------------------------------------------------

def build_ppo_env(base_env, seed: int, csv_writer, run_metadata: dict):
    import gymnasium as gym

    class PPOEnv(gym.Wrapper):
        def __init__(self, wrapped):
            super().__init__(wrapped)
            self._episode_return = 0.0
            self._episode_steps = 0
            self._episode_idx = 0
            self._total_step = 0

        def reset(self, **kwargs):
            obs, info = self.env.reset(**kwargs)
            self._episode_return = 0.0
            self._episode_steps = 0
            return obs, info

        def step(self, action):
            obs, reward, terminated, truncated, info = self.env.step(action)
            self._episode_return += float(reward)
            self._episode_steps += 1
            self._total_step += 1
            if terminated or truncated:
                self._episode_idx += 1
                success = bool(reward > 0)
                if csv_writer is not None:
                    csv_writer.writerow({
                        "seed": run_metadata["seed"],
                        "config": run_metadata["config_name"],
                        "step": self._total_step,
                        "episode": self._episode_idx,
                        "return": round(self._episode_return, 6),
                        "shaped_return": round(self._episode_return, 6),
                        "success": int(success),
                        "steps": self._episode_steps,
                        "subgoal_advances": 0,
                        "replan_periodic": 0,
                        "replan_failure": 0,
                        "llm_calls": 0,
                        "llm_tokens_in": 0,
                        "llm_tokens_out": 0,
                        "llm_wallclock_sec": 0.0,
                    })
            return obs, reward, terminated, truncated, info

    return PPOEnv(base_env)

# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------

CSV_FIELDS = [
    "seed", "config", "step", "episode", "return", "shaped_return",
    "success", "steps", "subgoal_advances",
    "replan_periodic", "replan_failure",
    "llm_calls", "llm_tokens_in", "llm_tokens_out", "llm_wallclock_sec",
]

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--config", required=True)
    p.add_argument("--env", default=None, help="Override env id from config")
    p.add_argument("--steps", type=int, default=100_000)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--out", default=None, help="CSV log path (default: runs/{cfg}_{seed}.csv)")
    p.add_argument("--planner", default=None,
                   choices=[None, "mock", "ollama", "groq", "random"],
                   help="Override planner backend from config")
    p.add_argument("--dry-run", action="store_true",
                   help="Build everything, print structure, exit before training")
    args = p.parse_args()

    random.seed(args.seed)
    np.random.seed(args.seed)

    cfg = yaml.safe_load(Path(args.config).read_text(encoding="utf-8"))
    if args.env:
        cfg["env"]["id"] = args.env
    env_id = cfg["env"]["id"]
    wraps = cfg["env"].get("wrap", [])

    out_path = Path(args.out) if args.out else Path("runs") / f"{cfg['name']}_seed{args.seed}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    csv_file = open(out_path, "w", newline="", encoding="utf-8")
    csv_writer = csv.DictWriter(csv_file, fieldnames=CSV_FIELDS)
    csv_writer.writeheader()

    run_metadata = {"seed": args.seed, "config_name": cfg["name"]}

    print(f"[train] config={cfg['name']}  env={env_id}  steps={args.steps}  seed={args.seed}")
    print(f"[train] logging to {out_path}")

    base = build_base_env(env_id, wraps, args.seed)

    if cfg["policy_type"] == "pure_ppo":
        env = build_ppo_env(base, args.seed, csv_writer, run_metadata)
    elif cfg["policy_type"] == "hybrid":
        planner = build_planner(cfg.get("planner", {}), override_backend=args.planner)
        encoder = build_encoder(cfg.get("encoder", {}))
        env = build_hybrid_env(base, cfg, planner, encoder, args.seed, csv_writer, run_metadata)
    else:
        raise ValueError(f"Unknown policy_type: {cfg['policy_type']}")

    from baselines.ppo import build_ppo
    model = build_ppo(env, cfg, seed=args.seed, verbose=0)

    if args.dry_run:
        print("[train] --dry-run: pipeline built, exiting before learn()")
        print(f"[train]   policy: {type(model.policy).__name__}")
        print(f"[train]   env obs space: {env.observation_space}")
        print(f"[train]   env action space: {env.action_space}")
        csv_file.close()
        return

    t0 = time.time()
    model.learn(total_timesteps=args.steps, progress_bar=False)
    dt = time.time() - t0
    print(f"[train] done  wallclock={dt:.1f}s  ({args.steps/max(dt,1e-9):.1f} steps/s)")

    csv_file.close()

if __name__ == "__main__":
    sys.path.insert(0, str(Path(__file__).parent))
    main()


### E2. Configs (inlined from `baselines/configs/*.yaml`)


In [ ]:
PPO_CONFIG = {
    "name": "ppo",
    "policy_type": "pure_ppo",
    "env": {"id": "MiniGrid-DoorKey-6x6-v0", "wrap": ["flat_obs"]},
    "ppo": {"learning_rate": 3.0e-4, "n_steps": 256, "batch_size": 64,
            "n_epochs": 4, "clip_range": 0.2, "gae_lambda": 0.95,
            "gamma": 0.99, "ent_coef": 0.01, "vf_coef": 0.5,
            "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [64, 64]}},
}

def _make_hybrid_config():
    # Builds the hybrid pilot config using whichever LLM backend is available.
    if LLM_BACKEND == "groq":
        planner = {"backend": "groq", "model": "llama-3.1-8b-instant",
                   "temperature": 0.0, "max_plan_length": 12,
                   "max_subgoal_tokens": 32, "environment_name": "MiniGrid"}
    else:  # ollama or mock
        planner = {"backend": "ollama", "model": "qwen2.5:14b",
                   "temperature": 0.0, "max_plan_length": 12,
                   "max_subgoal_tokens": 32, "environment_name": "MiniGrid"}
    return {
        "name": "hybrid_pilot",
        "policy_type": "hybrid",
        "env": {"id": "MiniGrid-DoorKey-6x6-v0", "wrap": ["flat_obs"]},
        "planner": planner,
        "shaping": {"enabled": False, "magnitude": 0.0, "gamma": 0.99},
        "schedule": {"done_oracle": "env_event"},
        "replan": {"period_H": 1000, "budget_B": 0},
        "encoder": {"backend": "sentence_transformer",
                    "model_name": "sentence-transformers/all-MiniLM-L6-v2"},
        "ppo": PPO_CONFIG["ppo"],
    }

HYBRID_PILOT_CONFIG = _make_hybrid_config()


### E3. Pilot analysis (IQM + bootstrap CI)


In [ ]:
"""Pilot analysis: read pilot/logs/*.csv → produce pilot/pilot_output.md with
a compact Appendix-C-ready table.

Deliberately does not use eval/report.py's default formatter, because the pilot
needs an extra column reporting subgoal / replan counts and per-config wall
clock summary that the general reporter does not carry.

Usage:
    python pilot/analysis.py --logs pilot/logs --out pilot/pilot_output.md
"""
from __future__ import annotations

import argparse
import glob
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# (defined in an earlier cell)

def load(logs_dir: str) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(logs_dir, "*.csv")))
    if not files:
        raise SystemExit(f"[analysis] no CSVs found under {logs_dir}")
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        if df.empty:
            continue
        df["source"] = os.path.basename(f)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

def per_seed(df: pd.DataFrame, tail_frac: float = 0.25) -> pd.DataFrame:
    rows = []
    for (cfg, seed), g in df.groupby(["config", "seed"]):
        g = g.sort_values("episode")
        n = len(g)
        if n == 0:
            continue
        tail = max(1, int(n * tail_frac))
        tail_g = g.tail(tail)
        rolling = rolling_success_rate(g["success"].to_numpy(), window=50)
        s2t = steps_to_threshold(g["step"].to_numpy(), rolling, threshold=0.5)
        rows.append({
            "config": cfg,
            "seed": int(seed),
            "n_episodes": int(n),
            "final_success_rate": float(tail_g["success"].mean()),
            "final_return": float(tail_g["return"].mean()),
            "mean_steps_per_ep": float(g["steps"].mean()),
            "steps_to_50pct": float(s2t),
            "subgoal_advances_total": int(g["subgoal_advances"].sum()) if "subgoal_advances" in g else 0,
            "replans_periodic_total": int(g["replan_periodic"].max()) if "replan_periodic" in g else 0,
            "replans_failure_total": int(g["replan_failure"].max()) if "replan_failure" in g else 0,
            "llm_calls": int(g["llm_calls"].max()) if "llm_calls" in g else 0,
            "llm_tokens_out": int(g["llm_tokens_out"].max()) if "llm_tokens_out" in g else 0,
            "llm_wallclock_sec": float(g["llm_wallclock_sec"].max()) if "llm_wallclock_sec" in g else 0.0,
        })
    return pd.DataFrame(rows).sort_values(["config", "seed"]).reset_index(drop=True)

def aggregate(ps: pd.DataFrame, n_boot: int = 2000) -> pd.DataFrame:
    rng = np.random.default_rng(0)
    metrics = ["final_success_rate", "final_return", "mean_steps_per_ep",
               "steps_to_50pct"]
    rows = []
    for cfg, g in ps.groupby("config"):
        row = {"config": cfg, "n_seeds": int(len(g))}
        for m in metrics:
            vals = g[m].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                row[f"{m}_iqm"] = float("nan")
                row[f"{m}_lo"] = float("nan")
                row[f"{m}_hi"] = float("nan")
                continue
            row[f"{m}_iqm"] = iqm(vals)
            lo, hi = bootstrap_iqm_ci(vals, n_boot=n_boot, rng=rng)
            row[f"{m}_lo"] = lo
            row[f"{m}_hi"] = hi
        row["llm_calls_total"] = int(g["llm_calls"].sum())
        row["llm_tokens_out_total"] = int(g["llm_tokens_out"].sum())
        row["llm_wallclock_hours"] = float(g["llm_wallclock_sec"].sum() / 3600.0)
        rows.append(row)
    return pd.DataFrame(rows)

def fmt_iqm(v, lo, hi, pct=False, digits=3):
    if not np.isfinite(v):
        return "—"
    if pct:
        return f"{v*100:.1f}\\% [{lo*100:.1f}, {hi*100:.1f}]"
    if not np.isfinite(lo) or not np.isfinite(hi):
        return f"{v:.{digits}f}"
    return f"{v:.{digits}f} [{lo:.{digits}f}, {hi:.{digits}f}]"

def format_markdown(agg: pd.DataFrame, ps: pd.DataFrame, env_id: str,
                    total_steps_per_seed: int, n_seeds: int) -> str:
    lines = [
        "### Pipeline validation pilot (Appendix C)",
        "",
        f"Environment: `{env_id}`. Per-seed budget: {total_steps_per_seed:,} "
        f"environment steps. Seeds: {n_seeds}. Reporting: IQM $\\pm$ 95\\% bootstrap CI "
        f"per Agarwal et al.\\ 2021, following the paper's committed statistical protocol.",
        "",
        "| Config | n | Success rate | Final return | Mean steps/ep | Steps to 50\\% | LLM calls / seed |",
        "|---|---:|---|---|---|---|---:|",
    ]
    for _, row in agg.iterrows():
        cfg = row["config"]
        n = int(row["n_seeds"])
        sr = fmt_iqm(row["final_success_rate_iqm"],
                     row["final_success_rate_lo"], row["final_success_rate_hi"], pct=True)
        fr = fmt_iqm(row["final_return_iqm"],
                     row["final_return_lo"], row["final_return_hi"])
        ms = fmt_iqm(row["mean_steps_per_ep_iqm"],
                     row["mean_steps_per_ep_lo"], row["mean_steps_per_ep_hi"])
        s2t = row["steps_to_50pct_iqm"]
        s2t_txt = f"{int(s2t):,}" if np.isfinite(s2t) else "not reached"
        llm_per_seed = int(row["llm_calls_total"] / max(1, n))
        lines.append(f"| `{cfg}` | {n} | {sr} | {fr} | {ms} | {s2t_txt} | {llm_per_seed:,} |")

    total_llm_hr = agg["llm_wallclock_hours"].sum()
    total_llm_calls = agg["llm_calls_total"].sum()
    total_llm_tokens = agg["llm_tokens_out_total"].sum()

    lines += [
        "",
        f"*LLM budget across all pilot runs: {total_llm_calls:,} calls, "
        f"{total_llm_tokens:,} output tokens, {total_llm_hr:.2f} hours of Ollama wall clock.*",
    ]
    return "\n".join(lines) + "\n"

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--logs", default="pilot/logs")
    ap.add_argument("--out", default="pilot/pilot_output.md")
    ap.add_argument("--env", default="MiniGrid-DoorKey-6x6-v0")
    args = ap.parse_args()

    df = load(args.logs)
    ps = per_seed(df)
    agg = aggregate(ps)
    steps_per_seed = int(df.groupby(["config", "seed"])["step"].max().max())
    n_seeds = int(df.groupby("config")["seed"].nunique().max())
    md = format_markdown(agg, ps, args.env, steps_per_seed, n_seeds)
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(md, encoding="utf-8")
    Path(args.out).with_suffix(".json").write_text(
        pd.concat([agg, ps], keys=["aggregate", "per_seed"]).to_json(indent=2),
        encoding="utf-8",
    )
    print(md)
    print(f"[analysis] wrote {args.out}")

if __name__ == "__main__":
    main()


**Run the demo.** This kicks off 3 seeds of PPO followed by 3 seeds of `hybrid_pilot` (the hybrid seeds run only if `LLM_BACKEND` is not `mock`). Long-running. Feel free to interrupt after any seed and rerun the analysis cell to see partial results.


In [ ]:
# --- Set up per-run temp dir for logs ---
import tempfile as _tempfile
from pathlib import Path as _Path

_pilot_dir = _Path(_tempfile.mkdtemp(prefix="pilot_"))
(_pilot_dir / "logs").mkdir()
print(f"Pilot logs will be written to: {_pilot_dir / 'logs'}")

# --- Save configs as YAML files (train.py reads YAML) ---
import yaml as _yaml
(_pilot_dir / "ppo.yaml").write_text(_yaml.safe_dump(PPO_CONFIG), encoding="utf-8")
(_pilot_dir / "hybrid.yaml").write_text(_yaml.safe_dump(HYBRID_PILOT_CONFIG), encoding="utf-8")

# --- Run PPO baseline (3 seeds) ---
import sys as _sys, time as _time
_argv_backup = _sys.argv[:]
for _seed in [0, 1, 2]:
    _out_csv = _pilot_dir / "logs" / f"ppo_seed{_seed}.csv"
    print(f"\n>>> PPO seed={_seed} ...", flush=True)
    _sys.argv = ["train.py", "--config", str(_pilot_dir / "ppo.yaml"),
                 "--env", "MiniGrid-DoorKey-6x6-v0",
                 "--steps", "30000", "--seed", str(_seed),
                 "--out", str(_out_csv)]
    _t0 = _time.time()
    try:
        main()
    except Exception as _e:
        print(f"    error: {_e}")
    print(f"    done in {_time.time()-_t0:.0f}s")

# --- Run hybrid_pilot (3 seeds, only if Ollama is up) ---
if not OLLAMA_AVAILABLE:
    print("\nSkipping hybrid seeds: Ollama with qwen2.5:14b not available.")
else:
    for _seed in [0, 1, 2]:
        _out_csv = _pilot_dir / "logs" / f"hybrid_pilot_seed{_seed}.csv"
        print(f"\n>>> HYBRID seed={_seed} ...", flush=True)
        _sys.argv = ["train.py", "--config", str(_pilot_dir / "hybrid.yaml"),
                     "--env", "MiniGrid-DoorKey-6x6-v0",
                     "--steps", "30000", "--seed", str(_seed),
                     "--out", str(_out_csv)]
        _t0 = _time.time()
        try:
            main()
        except Exception as _e:
            print(f"    error: {_e}")
        print(f"    done in {_time.time()-_t0:.0f}s")

_sys.argv = _argv_backup
print("\nPilot training complete. Logs at:", _pilot_dir / "logs")


**Aggregate results (IQM + 95% bootstrap CI):**


In [ ]:
# Analysis reads all *.csv under the logs dir and produces the Markdown table.
# Works whether or not the training cell above was run:
#   - If run: uses _pilot_dir from the training cell.
#   - If skipped: point _pilot_dir at any directory that contains pilot CSVs.
import sys as _sys, tempfile as _tempfile
from pathlib import Path as _Path

if "_pilot_dir" not in dir():
    # Fallback: try the reference-repo location, then abort with a clear message.
    _candidate = _Path("./pilot/logs")
    if _candidate.exists() and any(_candidate.glob("*.csv")):
        _pilot_dir = _candidate.parent
        print(f"Using existing pilot logs at {_candidate.resolve()}")
    else:
        raise RuntimeError(
            "No pilot logs found. Run the training cell above first, "
            "or set _pilot_dir to a directory containing pilot/logs/*.csv."
        )

_argv_backup = _sys.argv[:]
_out_md = _pilot_dir / "pilot_output.md"
_sys.argv = ["analysis.py", "--logs", str(_pilot_dir / "logs"),
             "--out", str(_out_md),
             "--env", "MiniGrid-DoorKey-6x6-v0"]
try:
    main()
finally:
    _sys.argv = _argv_backup


## Notes

- Every number in Paper 1 traces to code in this notebook. No hand-entered values.
- The full source tree (individual `.py` files, YAML configs, pytest suite) is easier to extend than editing inlined notebook cells. See the `paper1_framework` repository on GitHub, linked from the paper.
- Bug reports and improvements: open an issue on the repository or email `christophe.hounwanou@aims.ac.rw`.

---

*Generated from the reference implementation. Update the underlying `.py` files, then regenerate.*
